# How good is the model at word problems before finetuning?

## Answer Checker

In [1]:
# basic solution checkers
import re
import multiprocessing as mp


def final_answer_formatting(generated_text):
    """ check whether the answer ends with
        'The final answer is: __'
        and returns the imputed answer

        formatting codes: 0 - no good,
                          1 - contains answer,
                          2 - ends with 'The final answer is: __'
    """
    output = {'good_formatting': 0, 'perfect_formatting': 0, 'answer': ''}

    last_line = generated_text.split('\n')[-1]

    last_line_numbers = re.findall('[0-9]+', last_line)

    if len(last_line_numbers) > 0:
        output['answer'] = last_line_numbers[-1]

        if last_line.startswith('The final answer is:'):
            output['perfect_formatting'] = 1
            output['good_formatting'] = 1
        elif 'answer' in last_line.lower():
            output['good_formatting'] = 1

    return output


def COT_formatting(generated_text):
    pass


def results_worker(queue, output):
    """ worker for answer checker """
    while True:
        results = queue.get()
        if results is None:  # signal that the testing is done
            break

        samples, ground_truth = results
        output['num_inputs'] += 1

        correct_count = 0
        for sample in samples:
            answer_check = final_answer_formatting(sample['generated_text'])
            for key in ['good_formatting', 'perfect_formatting']:
                output[key] += answer_check[key]

            if answer_check['answer'] == ground_truth:
                correct_count += 1

        if correct_count > 0:
            output['passOf8'] += 1
        if correct_count > 4:
            output['majorityOf8'] += 1


class AnswerChecker(object):
    """ Multiprocessing answer checker
    """
    def __init__(self):
        self.queue = mp.Queue()
        self.results = mp.Manager().dict()
        self.results['good_formatting'] = 0
        self.results['perfect_formatting'] = 0
        self.results['majorityOf8'] = 0
        self.results['passOf8'] = 0
        self.results['num_inputs'] = 0

        self.worker = mp.Process(target=results_worker, args=(self.queue, self.results))
        self.worker.start()

    def add_result(self, results):
        self.queue.put(results)

    def shutdown(self):
        self.queue.put(None)

    def summarize(self):
        results = {key: self.results[key] / self.results['num_inputs']
                   for key in self.results.keys()}
        results.pop('num_inputs')
        results['num_samples'] = self.results['num_inputs']

        results['good_formatting'] = results['good_formatting'] / 8
        results['perfect_formatting'] = results['perfect_formatting'] / 8

        return results


In [2]:
answer_checker = AnswerChecker()

## Model and dataset setup

In [3]:
from transformers import pipeline
model = pipeline("text-generation", model="meta-llama/Llama-3.2-1B-Instruct",
    device_map='auto', do_sample=True, temperature=1.,
)
model.tokenizer.pad_token_id = model.model.config.eos_token_id[2]
_ = model.model.eval()

Device set to use cuda:0


In [4]:
from datasets import load_dataset
questions = load_dataset("openai/gsm8k", "main")

prompt = "Please solve the following math problem, reasoning step by step and stating the final answer at the end. Question: {question} "

def process_datapoint(datapoint):
    answer = datapoint.pop('answer').split('####')[-1].strip()
    question = prompt.format(**datapoint) 
    return {'question': question, 'gt': answer}

questions = questions['train'].map(process_datapoint)

In [5]:
import torch

batch_size=32
with torch.no_grad():
    for i in range(len(questions) // batch_size + 1):
        batch = questions[i*batch_size:(i+1)*batch_size]
    
        outs = model(batch['question'], batch_size=batch_size, num_return_sequences=8)
        
        for result in zip(outs, batch['gt']):
            answer_checker.add_result(result)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_to

OutOfMemoryError: CUDA out of memory. Tried to allocate 974.00 MiB. GPU 0 has a total capacity of 19.70 GiB of which 400.88 MiB is free. Process 3187848 has 19.30 GiB memory in use. Of the allocated memory 13.55 GiB is allocated by PyTorch, and 5.53 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [6]:
answer_checker.shutdown()
answer_checker.worker.join()
answer_checker.summarize()

{'good_formatting': 0.5442457932692307,
 'perfect_formatting': 0.33852914663461536,
 'majorityOf8': 0.00811298076923077,
 'passOf8': 0.06610576923076923,
 'num_samples': 3328}

Results from a run in the cloud:
```
{'good_formatting': 0.5442457932692307,
 'perfect_formatting': 0.33852914663461536,
 'majorityOf8': 0.00811298076923077,
 'passOf8': 0.06610576923076923,
 'num_samples': 3328}
```

These results make setting appear promising: we get many results with good formatting based on the prompt alone, but there is little in the way of accuracy. I need to double check this: it appears to me to be too low accuracy compared to what I've seen in my fiddling.

Note for implementation: the GPU ran out of memory, which killed the run